# Sequential text pre-processing

In [44]:
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('./data/wyoming.csv')
# df = pd.read_csv('./data/new-england.csv')
df.shape

(234655, 7)

In [45]:
df.loc[0,['text']]

text    When knowledge is key and kindness matters, Niki Morrison is the right combination every time.
Name: 0, dtype: object

In [46]:
df.head(2)

,gmap_id,rating,time,text,category,latitude,longitude
0,0x8758dd1ca83449d9:0xb6156dcfc5e04c9b,5,1602893531994,"When knowledge is key and kindness matters, Niki Morrison is the right combination every time.","['Real estate agency', 'Commercial real estate agency', 'Real estate agents', 'Real estate consu...",43.022906,-108.386042
1,0x8758dd1ca83449d9:0xb6156dcfc5e04c9b,5,1575991509006,"The entire team is outstanding! They are professional, knowledgeable, and friendly. The only cho...","['Real estate agency', 'Commercial real estate agency', 'Real estate agents', 'Real estate consu...",43.022906,-108.386042


In [47]:
# Added for 'new-england.csv'
df = df[df['text'].apply(lambda x: isinstance(x, str))]

In [48]:
df.shape

(234655, 7)

### Tokenize

The Keras `Tokenizer` seems to be the most modern all-in-one solution

In [49]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text'])
X_seq = tokenizer.texts_to_sequences(df['text'])

In [50]:
len(tokenizer.word_index) # Maximum vocabulary size (may useful for setting LSTM model vocab_size)

53667

### Verify output

A review sequence of text is processed (lower case, remove punctuation, etc.) and converted to tokens.

Tokens are converted to numbers. The maximum number is the vocabulary size (total number of unique tokens): `len(tokenizer.word_index)`

In [51]:
X_seq[0]

[64, 1225, 9, 1790, 3, 3082, 5095, 1, 1, 9, 2, 153, 2939, 163, 49]

`tokenizer.index_word` is the dictionary that links an encoding number to a particular token

In [52]:
tokenizer.index_word

{1: '<OOV>',
 2: 'the',
 3: 'and',
 4: 'a',
 5: 'to',
 6: 'i',
 7: 'great',
 8: 'was',
 9: 'is',
 10: 'food',
 11: 'good',
 12: 'of',
 13: 'in',
 14: 'for',
 15: 'it',
 16: 'service',
 17: 'they',
 18: 'place',
 19: 'very',
 20: 'you',
 21: 'my',
 22: 'but',
 23: 'are',
 24: 'have',
 25: 'this',
 26: 'with',
 27: 'friendly',
 28: 'staff',
 29: 'not',
 30: 'on',
 31: 'nice',
 32: 'we',
 33: 'that',
 34: 'always',
 35: 'there',
 36: 'had',
 37: 'at',
 38: 'so',
 39: 'love',
 40: 'were',
 41: 'be',
 42: 'get',
 43: 'best',
 44: 'here',
 45: 'as',
 46: 'clean',
 47: 'all',
 48: 'out',
 49: 'time',
 50: 'go',
 51: 'if',
 52: "it's",
 53: 'people',
 54: 'like',
 55: 'me',
 56: 'one',
 57: 'just',
 58: 'our',
 59: 'their',
 60: 'what',
 61: 'store',
 62: 'or',
 63: 'awesome',
 64: 'when',
 65: 'prices',
 66: 'amazing',
 67: 'your',
 68: 'from',
 69: 'up',
 70: 'will',
 71: 'really',
 72: 'an',
 73: 'helpful',
 74: 'well',
 75: 'no',
 76: 'by',
 77: 'has',
 78: 'excellent',
 79: 'fast',
 80: '

In [53]:
[tokenizer.index_word[i] for i in X_seq[0]]

['when',
 'knowledge',
 'is',
 'key',
 'and',
 'kindness',
 'matters',
 '<OOV>',
 '<OOV>',
 'is',
 'the',
 'right',
 'combination',
 'every',
 'time']

### Pad sequences

Make observations of identical size.

Note: Padding increases data size, so saved files take up more space. However, train-test split on a variable-length list object is a hassle (Alteratively, add text pre-processing to model file without saving to disk.)

Note: There is some potential for "leakage" between training and test data if padding before splitting.

In [54]:
lengths = [len(seq) for seq in X_seq]
print(f'Max length: {np.max(lengths)}')
print(f'Average length: {np.mean(lengths)}')
print(f'Quantiles: {np.quantile(lengths, [0.01, 0.10, 0.25, 0.5, 0.75, 0.90, 0.99])}')

Max length: 788
Average length: 17.56331209648207
Quantiles: [  1.   2.   5.   9.  20.  40. 125.]


In [55]:
max_length = 50
X = pad_sequences(X_seq, maxlen=max_length, padding='post', truncating='post')

In [56]:
del X_seq

In [57]:
X[0:2]

array([[  64, 1225,    9, 1790,    3, 3082, 5095,    1,    1,    9,    2,
         153, 2939,  163,   49,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0],
       [   2,  742,  888,    9,  533,   17,   23,  320,  260,    3,   27,
           2,   87,  662,   13, 2787, 1412,  623,    1,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0]], dtype=int32)

In [58]:
X.shape

(234655, 50)

The resulting data is 2-dimensional:

* Rows are observations

* Columns are numerically-indexed tokens, padded with zeros, set to length 50 (50 tokens max in a sequence)

* Each numerically-indexed token is an integer representing a token, with max size equal to vocabulary size

### Train-test split

In [59]:
y = df['rating'] - 1 # Classification should be 0-4 indexed

In [60]:
del df

In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

In [62]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((187724, 50), (187724,), (46931, 50), (46931,))

In [ ]:
np.save('./data/X_train_seq.npy', X_train)
np.save('./data/y_train_seq.npy', y_train)

np.save('./data/X_test_seq.npy', X_test)
np.save('./data/y_test_seq.npy', y_test)

# GloVe Embedding Matrix

From sequential data to embedding vectors:

* If the data has been properly tokenized, it's in 2D format, where each row is a text review, and each column is a token converted to an integer (token index) with the maximum value equal to the vocabulary size

* The embedding matrix then converts each token (in integer form) to a vector of chosen dimensionality (e.g. 50)

* The embedding matrix is sort of likea lookup table: give it an integer, and it returns the corresponding vector

In [64]:
filepath='/Users/andy/_data/glove.6B/glove.6B.50d.txt'
embeddings = {}

with open(filepath, 'r', encoding='utf-8') as file:
    for line in file:
        parts = line.split() # length is [0] word, [1:] embedding vector
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        embeddings[word] = vector

In [78]:
embeddings['germany'].shape, embeddings.get('germany').shape

((50,), (50,))

In [76]:
embedding_dim = 50  # Match the pre-trained embedding 
vocab_size = len(tokenizer.word_index) + 1

glove_embedding = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():
    vector = embeddings.get(word) # .get() does not return an error if an etry is not found, [] notation does
    if vector is not None:
        glove_embedding[idx] = vector


In [77]:
glove_embedding.shape

(53668, 50)

In [80]:
np.save('data/glove_embedding_matrix.npy', glove_embedding)